In [ ]:
#@title  Connect { display-mode: "form" }
# One click. Sets everything up.

import os, sys, subprocess, shutil, importlib.util, urllib.request, warnings
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as W

warnings.filterwarnings("ignore")

_S = globals().setdefault("_MS", {})
_S.setdefault("work", Path("/content/motionsalt"))
_S["work"].mkdir(parents=True, exist_ok=True)
_S.setdefault("ready", False)

_status = W.HTML()
_bar = W.IntProgress(value=0, min=0, max=5, bar_style="info",
                     layout=W.Layout(width="360px"))
_box = W.VBox([W.HTML(
    "<div style='font:600 15px system-ui;color:#111'>Setting up your workspace</div>"
    "<div style='font:13px system-ui;color:#666;margin-top:2px'>Takes a moment. You only do this once per session.</div>"
), _bar, _status])
display(_box)

def _tick(n, msg):
    _bar.value = n
    _status.value = f"<div style='font:13px system-ui;color:#444;margin-top:4px'>{msg}</div>"

def _done(msg, ok=True):
    _bar.value = _bar.max
    _bar.bar_style = "success" if ok else "danger"
    _status.value = (
        f"<div style='font:600 14px system-ui;color:{'#0a7d2c' if ok else '#b00020'};"
        f"margin-top:6px'>{msg}</div>"
    )

try:
    _tick(1, "Checking GPU…")
    try:
        import torch
        if not torch.cuda.is_available():
            _done("No GPU. Enable one: Runtime → Change runtime type → GPU.", ok=False)
            raise SystemExit
    except SystemExit:
        raise
    except Exception:
        pass

    _tick(2, "Installing system tools…")
    if not shutil.which("ffmpeg"):
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"],
                       check=False, capture_output=True)

    _tick(3, "Installing packages…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "pymongo", "dnspython", "gdown", "pycuda",
         "tensorrt==11.0.0.114", "opencv-python", "pillow", "pepedpid"],
        check=False, capture_output=True,
    )

    _tick(4, "Warming up…")

    def _locate():
        for base in [Path.cwd(), Path("/content"), Path("/content/upscale")]:
            c = base / "engine" / "config_utils.so"
            if c.exists() and c.stat().st_size > 10000:
                return c
        dest = _S["work"] / "._core"
        if dest.exists() and dest.stat().st_size > 10000:
            return dest
        raw = "https://raw.githubusercontent.com/motionssalt/upscale/main/engine/config_utils.so"
        req = urllib.request.Request(raw, headers={"User-Agent": "motionsalt"})
        with urllib.request.urlopen(req, timeout=60) as r, dest.open("wb") as f:
            shutil.copyfileobj(r, f)
        return dest

    _p = _locate()
    import importlib.machinery
    _loader = importlib.machinery.ExtensionFileLoader("config_utils", str(_p))
    _spec = importlib.util.spec_from_file_location("config_utils", str(_p), loader=_loader)
    if _spec is None:
        raise RuntimeError(f"Could not build a module spec for {_p} — file missing or unreadable.")
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules["config_utils"] = _mod
    _spec.loader.exec_module(_mod)

    _S["_core"] = _mod
    _S["ready"] = True

    _done("Ready. Go to the next cell.")
except SystemExit:
    pass
except Exception as _e:
    _done(f"Setup failed: {_e}", ok=False)
    raise



In [ ]:
#@title  Pick your file { display-mode: "form" }
import time
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as W

_S = globals().setdefault("_MS", {})
if not _S.get("ready"):
    raise SystemExit("Run the Connect cell first.")

_S["work"].joinpath("input").mkdir(parents=True, exist_ok=True)

_title = W.HTML(
    "<div style='font:600 15px system-ui;color:#111'>Pick your file</div>"
    "<div style='font:13px system-ui;color:#666;margin-top:2px'>"
    "Choose a video from your computer.</div>"
)

_uploader = W.FileUpload(
    accept="video/*",
    multiple=False,
    description="Choose video",
    button_style="primary",
    layout=W.Layout(width="180px"),
)

_info = W.HTML(
    "<div style='font:13px system-ui;color:#888;margin-top:8px'>No file selected yet.</div>"
)

_card = W.VBox(
    [_title, W.HBox([_uploader]), _info],
    layout=W.Layout(
        border="1px solid #e4e4e7",
        border_radius="10px",
        padding="16px 18px",
        width="520px",
    ),
)
display(_card)

def _on_upload(change):
    val = _uploader.value
    if not val:
        return
    if isinstance(val, dict):
        fname, meta = next(iter(val.items()))
        content = meta["content"] if isinstance(meta, dict) else meta
    else:
        meta = val[0]
        fname = meta["name"]
        content = meta["content"]
    dest = _S["work"] / "input" / fname
    with open(dest, "wb") as f:
        f.write(bytes(content))
    _S["input"] = dest
    mb = dest.stat().st_size / 1e6
    _info.value = (
        f"<div style='font:13px system-ui;color:#0a7d2c;margin-top:8px'>"
        f"<b>{fname}</b> &middot; {mb:.1f} MB &middot; ready to enhance</div>"
    )

_uploader.observe(_on_upload, names="value")

if _S.get("input") and Path(_S["input"]).exists():
    _p = Path(_S["input"])
    _info.value = (
        f"<div style='font:13px system-ui;color:#0a7d2c;margin-top:8px'>"
        f"<b>{_p.name}</b> &middot; {_p.stat().st_size/1e6:.1f} MB &middot; ready to enhance</div>"
    )


In [ ]:
#@title  Enhance { display-mode: "form" }
import os, sys, io, json, time, uuid, mimetypes, urllib.request, contextlib
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as W

_S = globals().setdefault("_MS", {})
if not _S.get("ready"):
    raise SystemExit("Run the Connect cell first.")
if not _S.get("input") or not Path(_S["input"]).exists():
    raise SystemExit("Pick a file in the previous cell first.")

_core = _S["_core"]

_MODELS = [
    "2x Xyether Anime Sharp",
    "2x Xyether Anime Soft",
    "1x Xyether Compression Remover",
    "1x Xyether Dark CC v1",
    "2x Xyether Dark cc V2",
    "2x xyether Dark Blue CC",
    "2x xyether White CC",
    "1x Xyether Tiktok CC (Soft)",
    "1x xyether tiktok cc (strong)",
    "1x Faster Xyether Compression Remover",
]

_QUALITY = {
    "Draft (fastest)":   {"speed": True,  "crf": 20},
    "Balanced":          {"speed": False, "crf": 18},
    "High (slower)":     {"speed": False, "crf": 14},
    "Max (slowest)":     {"speed": False, "crf": 10},
}

_model = W.Dropdown(options=_MODELS, value="2x Xyether Anime Soft",
                    description="Model", style={"description_width": "80px"},
                    layout=W.Layout(width="440px"))
_qual  = W.Dropdown(options=list(_QUALITY.keys()), value="Balanced",
                    description="Quality", style={"description_width": "80px"},
                    layout=W.Layout(width="300px"))
_force = W.Checkbox(value=False, description="Force 1080p", indent=False)
_codec = W.ToggleButtons(options=[("H.265", "hevc_nvenc"), ("H.264", "h264_nvenc")],
                         value="hevc_nvenc",
                         style={"button_width": "80px"})

_run = W.Button(description="Enhance",
                button_style="primary",
                icon="play",
                layout=W.Layout(width="160px", height="38px"))
_progress = W.HTML()
_log = W.Output(layout=W.Layout(max_height="180px", overflow="auto"))

_form = W.VBox([
    W.HTML("<div style='font:600 15px system-ui;color:#111'>Enhance</div>"
           "<div style='font:13px system-ui;color:#666;margin-top:2px'>"
           f"Input: <b>{Path(_S['input']).name}</b></div>"),
    _model,
    W.HBox([_qual, _force]),
    W.HBox([W.HTML("<div style='font:13px system-ui;color:#444;width:80px;padding-top:6px'>Codec</div>"), _codec]),
    W.HBox([_run, _progress]),
    _log,
], layout=W.Layout(
    border="1px solid #e4e4e7",
    border_radius="10px",
    padding="16px 18px",
    width="620px",
))
display(_form)

def _find_output(in_path: Path, t0: float):
    dirs = {in_path.parent, Path("/content"), _S["work"], _S["work"] / "output"}
    best = None
    for d in dirs:
        if not d.exists():
            continue
        for f in d.iterdir():
            try:
                if not f.is_file() or f == in_path:
                    continue
                if f.suffix.lower() not in {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}:
                    continue
                if f.stat().st_mtime < t0 - 1:
                    continue
                if best is None or f.stat().st_mtime > best.stat().st_mtime:
                    best = f
            except OSError:
                continue
    return best

def _tmpfiles_upload(path: Path) -> str:
    boundary = f"----ms-{uuid.uuid4().hex}"
    ctype, _ = mimetypes.guess_type(str(path))
    ctype = ctype or "application/octet-stream"
    with open(path, "rb") as fh:
        body = fh.read()
    head = (
        f"--{boundary}\r\n"
        f'Content-Disposition: form-data; name="file"; filename="{path.name}"\r\n'
        f"Content-Type: {ctype}\r\n\r\n"
    ).encode()
    tail = f"\r\n--{boundary}--\r\n".encode()
    req = urllib.request.Request(
        "https://tmpfiles.org/api/v1/upload",
        data=head + body + tail,
        headers={
            "Content-Type": f"multipart/form-data; boundary={boundary}",
            "User-Agent": "motionsalt",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        payload = json.loads(r.read().decode("utf-8"))
    url = (payload.get("data") or {}).get("url", "")
    if not url:
        raise RuntimeError("share upload returned no URL")
    if "/dl/" not in url:
        url = url.replace("tmpfiles.org/", "tmpfiles.org/dl/", 1)
    return url

def _on_run(_btn):
    _run.disabled = True
    _model.disabled = _qual.disabled = _force.disabled = _codec.disabled = True
    _progress.value = "<span style='font:13px system-ui;color:#444;margin-left:12px'>Enhancing…</span>"
    _log.clear_output()

    in_path = Path(_S["input"])
    q = _QUALITY[_qual.value]
    t0 = time.time()

    try:
        with _log:
            _core.run_processing(
                video_path=str(in_path),
                model_choice=_model.value,
                Speed_Boost=q["speed"],
                Force_1080p=_force.value,
                codec=_codec.value,
                crf_value=q["crf"],
                auto_download=False,
            )
        elapsed = time.time() - t0
        out = _find_output(in_path, t0)
        if out is None:
            _progress.value = ("<span style='font:13px system-ui;color:#b00020;margin-left:12px'>"
                               "Finished, but no output file was found.</span>")
            _run.disabled = False
            _model.disabled = _qual.disabled = _force.disabled = _codec.disabled = False
            return
        _S["output"] = out
        _S["elapsed"] = elapsed
        _progress.value = (
            f"<span style='font:13px system-ui;color:#0a7d2c;margin-left:12px'>"
            f"Done in {elapsed:.0f}s &middot; <b>{out.name}</b> &middot; "
            f"{out.stat().st_size/1e6:.1f} MB &middot; open the next cell to download."
            f"</span>"
        )
        # Kick off share link in background-ish (blocking, but small text update).
        try:
            _S["share_url"] = _tmpfiles_upload(out)
            _S["share_created_at"] = time.time()
        except Exception:
            _S["share_url"] = None
    except Exception as e:
        _progress.value = f"<span style='font:13px system-ui;color:#b00020;margin-left:12px'>Failed: {e}</span>"
    finally:
        _run.disabled = False
        _model.disabled = _qual.disabled = _force.disabled = _codec.disabled = False

_run.on_click(_on_run)


In [ ]:
#@title  Download { display-mode: "form" }
from pathlib import Path
from IPython.display import display, HTML, Javascript
import ipywidgets as W

_S = globals().setdefault("_MS", {})
out = _S.get("output")
if not out or not Path(out).exists():
    display(HTML(
        "<div style='font:13px system-ui;color:#666;padding:14px 16px;"
        "border:1px dashed #d4d4d8;border-radius:10px;width:520px'>"
        "Run the Enhance cell first — your file will appear here."
        "</div>"
    ))
else:
    out = Path(out)
    size_mb = out.stat().st_size / 1e6
    share = _S.get("share_url")

    _title = W.HTML(
        "<div style='font:600 15px system-ui;color:#111'>Download</div>"
        f"<div style='font:13px system-ui;color:#666;margin-top:2px'>"
        f"<b>{out.name}</b> &middot; {size_mb:.1f} MB</div>"
    )

    _dl = W.Button(description="Download to my computer",
                   button_style="primary", icon="download",
                   layout=W.Layout(width="240px", height="38px"))
    _dl_status = W.HTML()

    def _do_download(_b):
        try:
            from google.colab import files as _cf
            _cf.download(str(out))
            _dl_status.value = "<span style='font:13px system-ui;color:#0a7d2c;margin-left:10px'>Started.</span>"
        except Exception as e:
            _dl_status.value = f"<span style='font:13px system-ui;color:#b00020;margin-left:10px'>{e}</span>"
    _dl.on_click(_do_download)

    if share:
        _link = W.HTML(
            "<div style='font:13px system-ui;color:#444;margin-top:14px'>"
            "Shareable link (valid ~1 hour):</div>"
            f"<div style='margin-top:4px'><a href='{share}' target='_blank' "
            "style='font:13px ui-monospace,Menlo,monospace;color:#1a56db;"
            f"word-break:break-all'>{share}</a></div>"
        )
    else:
        _link = W.HTML(
            "<div style='font:13px system-ui;color:#888;margin-top:14px'>"
            "Shareable link unavailable — use direct download.</div>"
        )

    _card = W.VBox(
        [_title, W.HBox([_dl, _dl_status]), _link],
        layout=W.Layout(
            border="1px solid #e4e4e7",
            border_radius="10px",
            padding="16px 18px",
            width="620px",
        ),
    )
    display(_card)
